# Notebook 02: Feature Extraction

## Goal
For each prompt in our taxonomy, extract the SAE feature activation vector 
from layers 6, 12, and 20 of Gemma 2 2B. Save results for analysis in Notebook 03.

## What we're computing
For each (prompt, layer) pair:
1. Run the prompt through Gemma 2 2B
2. Grab the residual stream at the target layer, last token position → vector of shape `[2304]`
3. Project through the Gemma Scope SAE → feature activations of shape `[16384]`

**Result**: A tensor of shape `[6 categories × 20 prompts, 16384 features]` per layer.

In [1]:
import torch
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
from transformer_lens import HookedTransformer
from sae_lens import SAE

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

RESULTS_DIR = Path('../results/feature_activations')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load config saved by notebook 01 — do not hardcode model or layers
with open('../results/config.json') as f:
    cfg = json.load(f)

MODEL_NAME    = cfg['model_name']
MODEL_DEVICE  = cfg['model_device']
MODEL_DTYPE   = cfg['model_dtype']
TARGET_LAYERS = cfg['target_layers']
SAE_RELEASE   = cfg['sae_release']
SAE_WIDTH     = cfg['sae_width']

# GPT-2 SAEs (gpt2-small-res-jb) sit on hook_resid_pre;
# Gemma Scope SAEs sit on hook_resid_post
HOOK_SUFFIX = 'hook_resid_pre' if 'gpt2' in SAE_RELEASE else 'hook_resid_post'

print(f'Model:         {MODEL_NAME}')
print(f'Device:        {MODEL_DEVICE}')
print(f'Target layers: {TARGET_LAYERS}')
print(f'SAE release:   {SAE_RELEASE}')
print(f'Hook:          blocks.N.{HOOK_SUFFIX}')

Model:         gpt2
Device:        cuda
Target layers: [2, 6, 10]
SAE release:   gpt2-small-res-jb
Hook:          blocks.N.hook_resid_pre


In [2]:
print(f'Loading {MODEL_NAME} on {MODEL_DEVICE}...')

model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    dtype=MODEL_DTYPE,
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
    device=MODEL_DEVICE,
)
model.eval()
print('Model loaded.')

Loading gpt2 on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2 into HookedTransformer
Model loaded.


In [3]:
# Load one SAE per target layer
# SAE.from_pretrained now returns just the SAE object (no longer a 3-tuple)
saes = {}
for layer in TARGET_LAYERS:
    if 'gpt2' in SAE_RELEASE:
        sae_id = f'blocks.{layer}.hook_resid_pre'
    else:
        sae_id = f'layer_{layer}/width_16k/average_l0_71'

    print(f'Loading SAE layer {layer}  ({sae_id})...')
    sae = SAE.from_pretrained(
        release=SAE_RELEASE,
        sae_id=sae_id,
        device=MODEL_DEVICE,
    )
    sae.eval()
    saes[layer] = sae
    print(f'  {sae.cfg.d_sae} features')

print('\nAll SAEs loaded.')

Loading SAE layer 2  (blocks.2.hook_resid_pre)...


cfg.json: 0.00B [00:00, ?B/s]

C:\Users\mukho\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mukho\.cache\huggingface\hub\models--jbloom--GPT2-Small-SAEs-Reformatted. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed

sae_weights.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


sae_weights.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


sparsity.safetensors:   0%|          | 0.00/98.4k [00:00<?, ?B/s]

  24576 features

All SAEs loaded.


In [4]:
# Load prompt taxonomy
with open('../data/prompts.json') as f:
    prompt_taxonomy = json.load(f)

categories = list(prompt_taxonomy.keys())
print(f'Categories: {categories}')
print(f'Prompts per category: {len(prompt_taxonomy[categories[0]])}')

Categories: ['math', 'code', 'factual', 'creative', 'emotional', 'reasoning']
Prompts per category: 20


In [5]:
def extract_features(model, sae, prompt, layer):
    """
    Extract SAE feature activations for a single prompt at a given layer.
    Returns np.ndarray of shape [n_features] — sparse, mostly zeros.
    """
    tokens = model.to_tokens(prompt)
    hook_name = f'blocks.{layer}.{HOOK_SUFFIX}'

    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, names_filter=hook_name)

    # Last token position; cast to float32 — SAE encoder expects it
    resid = cache[hook_name][0, -1, :].float()

    with torch.no_grad():
        feature_acts = sae.encode(resid.unsqueeze(0)).squeeze(0)

    return feature_acts.cpu().numpy()

print('Extraction function defined.')

Extraction function defined.


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Main extraction loop
# Runs all prompts across all layers — this is the expensive step
# Expected time: ~2-5 min on GPU, ~30-60 min on CPU
# ─────────────────────────────────────────────────────────────────────────────

# results[layer][category] = np.ndarray of shape [n_prompts, n_features]
all_results = {layer: {} for layer in TARGET_LAYERS}

for layer in TARGET_LAYERS:
    sae = saes[layer]
    print(f'\n=== Layer {layer} ===')
    
    for category in categories:
        prompts = prompt_taxonomy[category]
        category_features = []
        
        for prompt in tqdm(prompts, desc=f'  {category}'):
            features = extract_features(model, sae, prompt, layer)
            category_features.append(features)
        
        # Stack: [n_prompts, n_features]
        all_results[layer][category] = np.stack(category_features)
        
        # Print sparsity stats
        mean_active = (all_results[layer][category] > 0).sum(axis=1).mean()
        print(f'    {category}: avg {mean_active:.1f} active features')
    
    # Save layer results to disk
    save_path = RESULTS_DIR / f'layer_{layer}.npz'
    np.savez(save_path, **all_results[layer])
    print(f'  Saved to {save_path}')

print('\n✓ Extraction complete.')


=== Layer 2 ===


  math: 100%|████████████████████████████████████████████████████████| 20/20 [00:01<00:00, 12.25it/s]


    math: avg 15.1 active features


  code: 100%|████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 23.49it/s]


    code: avg 54.0 active features


...

    creative: avg 332.4 active features


  emotional: 100%|███████████████████████████████████████████████████| 20/20 [00:00<00:00, 25.93it/s]


    emotional: avg 227.2 active features


  reasoning: 100%|███████████████████████████████████████████████████| 20/20 [00:00<00:00, 24.98it/s]

    reasoning: avg 155.2 active features
  Saved to ..\results\feature_activations\layer_10.npz

✓ Extraction complete.


In [7]:
print('Verifying saved results...')
categories = list(json.load(open('../data/prompts.json')).keys())
for layer in TARGET_LAYERS:
    loaded = np.load(RESULTS_DIR / f'layer_{layer}.npz')
    print(f'\nLayer {layer}:')
    for cat in categories:
        arr = loaded[cat]
        print(f'  {cat}: shape={arr.shape}, nonzero_mean={np.mean(arr > 0):.4f}')

print('\nVerification complete. Proceed to Notebook 03.')

Verifying saved results...

Layer 2:
  math: shape=(20, 24576), nonzero_mean=0.0006
  code: shape=(20, 24576), nonzero_mean=0.0022
  factual: shape=(20, 24576), nonzero_mean=0.0009
  creative: shape=(20, 24576), nonzero_mean=0.0090
  emotional: shape=(20, 24576), nonzero_mean=0.0008
  reasoning: shape=(20, 24576), nonzero_mean=0.0004

Layer 6:
  math: shape=(20, 24576), nonzero_mean=0.0014
  code: shape=(20, 24576), nonzero_mean=0.0026
  factual: shape=(20, 24576), nonzero_mean=0.0022
  creative: shape=(20, 24576), nonzero_mean=0.0053
  emotional: shape=(20, 24576), nonzero_mean=0.0021
  reasoning: shape=(20, 24576), nonzero_mean=0.0016

Layer 10:
  math: shape=(20, 24576), nonzero_mean=0.0053
  code: shape=(20, 24576), nonzero_mean=0.0059
  factual: shape=(20, 24576), nonzero_mean=0.0103
  creative: shape=(20, 24576), nonzero_mean=0.0135
  emotional: shape=(20, 24576), nonzero_mean=0.0092
  reasoning: shape=(20, 24576), nonzero_mean=0.0063

Verification complete. Proceed to Notebook 0